# 사람들은 왜 '말포이'보다 '악역영애 말포이'를 더 검색했을까? (재현 노트북)

DataPopcorn 테크블로그 글 [datapopcorn.ai/blog/ts-malfoy-search-data](https://datapopcorn.ai/blog/ts-malfoy-search-data)의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글에 나온 숫자와 차트 6개가 다시 만들어집니다.

- 데이터: 2026-10-23에 수집한 스냅샷(네이버 검색광고 키워드도구, 네이버 데이터랩 일간·주간·성별, pixiv 태그 검색 날짜별 총계)
- API 키 없이 실행됩니다. 직접 다시 수집하려면 맨 아래 **부록: 수집 코드**를 보세요.
- 그림 파일과 작품 본문은 싣지 않았습니다. 날짜별 개수와 검색 지수만 있습니다.

In [ ]:
# 0. 준비: 한글 폰트 + 데이터 내려받기 (로컬에서는 DATA_DIR 환경변수로 폴더 지정)
import os, sys, urllib.request, urllib.parse
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        os.environ['KFONT'] = f; break
FILES = ['searchad_monthly.txt', 'daily_compare.json', 'daily_gender_female.json', 'daily_gender_male.json', 'week_mal_ay.json', 'pixiv_counts.json']
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/ts-malfoy-search-data/data')
if BASE.startswith('http'):
    os.makedirs('data', exist_ok=True)
    for fn in FILES:
        urllib.request.urlretrieve(f'{BASE}/{urllib.parse.quote(fn)}', f'data/{fn}')
    RAW = 'data'
else:
    RAW = BASE
os.environ['RAW'] = RAW; os.environ['FIG'] = 'fig'; os.environ['OUT'] = 'summary.json'
print('데이터 폴더:', RAW, sorted(os.listdir(RAW)))

In [ ]:
import json,re,os,pandas as pd,numpy as np,matplotlib
matplotlib.use('Agg');import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
FONT=os.environ.get('KFONT')
fm.fontManager.addfont(FONT);fam=fm.FontProperties(fname=FONT).get_name()
plt.rcParams.update({'font.family':fam,'axes.unicode_minus':False,'figure.dpi':150,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
C={'main':'#E8590C','sub':'#1C7ED6','gray':'#868E96','light':'#CED4DA'}
RAW=os.environ.get('RAW','raw');FIG=os.environ.get('FIG','fig');os.makedirs(FIG,exist_ok=True)
S={}
def dl(f):
    j=json.load(open(f'{RAW}/{f}'));return pd.DataFrame({r['title']:pd.Series({pd.Timestamp(x['period']):x['ratio'] for x in r['data']}) for r in j['results']})
AY,MAL,TS,HP='악역영애 말포이','말포이','TS말포이','해리포터'

## ① 최근 30일 실제 검색 횟수 (Step 2)
키워드도구의 최근 30일 PC+모바일 검색수입니다. 띄어쓰기 없는 표기로 집계됩니다.

In [ ]:
# ---------- 1. 최근 30일 검색수 ----------
sa={}
for line in open(f'{RAW}/searchad_monthly.txt'):
    m=re.match(r'(\S+)\t월 (\d+)회',line)
    if m: sa[m.group(1)]=int(m.group(2))
S['searchad_30d']=sa;S['ratio_30d_AY_MAL']=round(sa['악역영애말포이']/sa['말포이'],2)
fig,ax=plt.subplots(figsize=(10,4.2))
ks=['악역영애말포이','말포이','TS말포이','여자말포이'];vals=[sa[k] for k in ks]
b=ax.barh(ks[::-1],vals[::-1],color=[C['light'],C['light'],C['sub'],C['main']])
for y,v in enumerate(vals[::-1]):ax.text(v+400,y,f'{v:,}회',va='center',fontsize=10)
ax.set_xlim(0,max(vals)*1.18);ax.set_xlabel('최근 30일 검색수 (PC+모바일)')
ax.set_title(f'① 팬픽 제목이 캐릭터 이름보다 {S["ratio_30d_AY_MAL"]}배 더 검색됐다\n네이버 최근 30일 검색수',loc='left')
fig.text(0.01,-0.03,'자료: 네이버 검색광고 키워드도구(띄어쓰기 없는 표기 기준, 최근 30일). 2026-10-23 조회',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig1_volume.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig1_volume.png'))

## ② 일별 급변점 (Step 3)
비교할 검색어는 **한 요청 안에서** 받은 지수끼리만 비교합니다. 값이 없는 날은 0으로 채웁니다.

In [ ]:
# ---------- 2. 일별 비교 ----------
d=dl('daily_compare.json').reindex(pd.date_range('2026-08-01','2026-10-22')).fillna(0)
peak=d[AY].idxmax();S['peak_day']=str(peak.date());S['peak_ratio_AY_MAL']=round(d.loc[peak,AY]/d.loc[peak,MAL],2)
S['first_jump_day']=str(d[d[AY]>5].index[0].date())
cross=d[(d.index>peak)&(d[MAL]>d[AY])].index;S['first_cross_after_peak']=str(cross[0].date())
base=d.loc['2026-09-01':'2026-09-27',MAL].mean();last7=d.loc['2026-10-16':'2026-10-22']
S['MAL_base_0901_0927']=round(base,2);S['MAL_last7_mean']=round(last7[MAL].mean(),2);S['MAL_last7_x_base']=round(last7[MAL].mean()/base,1)
S['AY_last7_pct_of_peak']=round(last7[AY].mean()/d[AY].max()*100,1)
w=d.loc['2026-09-30':'2026-10-16'];S['sum_ratio_AY_MAL_0930_1016']=round(w[AY].sum()/w[MAL].sum(),2)
p=d.loc['2026-09-10':]
fig,ax=plt.subplots(figsize=(12,4.6))
ax.plot(p.index,p[AY],color=C['main'],lw=2.4,label="'악역영애 말포이'")
ax.plot(p.index,p[MAL],color=C['sub'],lw=2.2,label="'말포이'")
ax.plot(p.index,p[TS],color=C['gray'],lw=1.4,ls='--',label="'TS말포이' 등")
ax.annotate(f"10/7 정점\n팬픽 제목이 {S['peak_ratio_AY_MAL']}배",xy=(peak,100),xytext=(pd.Timestamp('2026-09-14'),88),arrowprops=dict(arrowstyle='->',color=C['main']),color=C['main'],fontsize=10)
ax.annotate("9/30 첫 급등",xy=(pd.Timestamp('2026-09-30'),13.4),xytext=(pd.Timestamp('2026-09-15'),40),arrowprops=dict(arrowstyle='->',color=C['gray']),fontsize=9)
cx=pd.Timestamp(S['first_cross_after_peak']);ax.axvline(cx,color=C['gray'],lw=.8,ls=':');ax.text(cx,92,' 10/17부터\n 말포이가 다시 앞섬',fontsize=9,color=C['sub'])
ax.set_ylabel('검색지수 (기간 최고일=100)');ax.legend(frameon=False,loc='upper right',bbox_to_anchor=(1,0.75))
ax.set_title('② 팬픽 제목 검색은 9/30에 튀어 10/7에 정점, 열흘 뒤 캐릭터 이름에 다시 역전됐다\n네이버 데이터랩 일별 검색지수 (한 요청 안에서 비교)',loc='left')
fig.text(0.01,-0.03,"자료: 네이버 데이터랩 검색어트렌드(일간). 그룹: 말포이=말포이·드레이코 말포이 / 악역영애 말포이=띄어쓰기 두 표기 / TS말포이=TS말포이·TS 말포이·여자 말포이·여자말포이. 2026-10-23 조회",fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig2_daily.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig2_daily.png'))

## ③ 픽시브 그림 vs 국내 검색 (Step 4)
픽시브 날짜별 총계는 업로드 날짜 필터(`scd`,`ecd`)로 하루씩 받은 값입니다(일본 시간).

In [ ]:
# ---------- 3. 픽시브 그림 vs 국내 검색 ----------
px=json.load(open(f'{RAW}/pixiv_counts.json'))
mf=pd.Series({pd.Timestamp(k):v for k,v in px['daily']['Malfoid'].items()}).astype(float)
dr=pd.Series({pd.Timestamp(k):v for k,v in px['daily']['ドラコ・マルフォイ'].items()}).astype(float)
S['malfoid_first_day']=str(mf[mf>0].index[0].date());S['malfoid_first_day_n']=int(mf[mf>0].iloc[0])
S['malfoid_peak_day']=str(mf.idxmax().date());S['malfoid_peak_n']=int(mf.max());S['malfoid_sum_0901_1022']=int(mf.sum())
S['malfoid_last7_mean']=round(mf.iloc[-7:].mean(),1)
fig,ax=plt.subplots(figsize=(12,4.6));q=mf.loc['2026-09-15':]
ax.bar(q.index,q.values,color=C['main'],alpha=.85,label="픽시브 'Malfoid' 태그 그림 수(왼쪽)")
ax2=ax.twinx();ax2.plot(d.loc['2026-09-15':].index,d.loc['2026-09-15':,AY],color=C['sub'],lw=2.2,label="네이버 '악역영애 말포이' 검색지수(오른쪽)")
ax.set_ylabel('하루 업로드 수 (장)');ax2.set_ylabel('검색지수');ax2.spines['right'].set_visible(True)
ax.text(pd.Timestamp('2026-09-28'),12,' 9/28 첫 등장\n 2장',fontsize=9)
ax.set_title(f"③ 해외 그림이 먼저 늘고(9/28~), 국내 검색이 이틀 뒤(9/30) 따라 올랐다\n픽시브 'Malfoid' 일별 업로드(최고 {S['malfoid_peak_n']}장, {S['malfoid_peak_day'][5:].replace('-','/')}) vs 네이버 검색",loc='left')
h1,l1=ax.get_legend_handles_labels();h2,l2=ax2.get_legend_handles_labels();ax.legend(h1+h2,l1+l2,frameon=False,loc='upper left',bbox_to_anchor=(0,0.9))
fig.text(0.01,-0.03,'자료: pixiv 태그 검색(부분일치, 업로드 날짜 필터, 일본 시간 기준) / 네이버 데이터랩. 2026-10-23 조회. 시점이 겹친다는 것이지 인과를 확인한 것은 아님',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig3_pixiv_vs_search.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig3_pixiv_vs_search.png'))

## ④ 10년 기준선 (Step 5)
원래 있던 태그 'ドラコ・マルフォイ'의 월별 업로드 수입니다. 2026년 10월은 10/23 조회 시점까지입니다.

In [ ]:
# ---------- 4. 픽시브 드레이코 월별 ----------
m=pd.Series(px['monthly']['ドラコ・マルフォイ']).astype(float)
S['draco_monthly_mean_2016_01_2026_08']=round(m.loc[:'2026-08'].mean(),1);S['draco_monthly_max_before']=int(m.loc[:'2026-08'].max())
S['draco_2026_09']=int(m['2026-09']);S['draco_2026_10']=int(m['2026-10'])
S['draco_by_year']=m.groupby(m.index.str[:4]).sum().astype(int).to_dict()
fig,ax=plt.subplots(figsize=(12,4.4))
cols=[C['main'] if k=='2026-10' else C['light'] for k in m.index]
ax.bar(range(len(m)),m.values,color=cols,width=.9)
ticks=[i for i,k in enumerate(m.index) if k.endswith('-01')];ax.set_xticks(ticks);ax.set_xticklabels([m.index[i][:4] for i in ticks])
ax.text(len(m)-1,m['2026-10']+30,f"2026년 10월\n{S['draco_2026_10']:,}장",ha='right',color=C['main'],fontsize=10)
ax.text(5,200,f"2016.1~2026.8 월평균 {S['draco_monthly_mean_2016_01_2026_08']}장 (최대 {S['draco_monthly_max_before']}장)",fontsize=10,color=C['gray'])
ax.set_ylabel('월별 업로드 수 (장)')
ax.set_title(f"④ 픽시브 '드레이코 말포이' 그림, 10년 동안 월평균 8장 안팎 → 이번 10월에만 {S['draco_2026_10']:,}장\n픽시브 태그 'ドラコ・マルフォイ' 월별 업로드 수",loc='left')
fig.text(0.01,-0.03,'자료: pixiv 태그 검색(부분일치, 월 단위 업로드 날짜 필터). 2026년 10월은 23일 조회 시점까지. 태그를 달지 않은 그림은 빠진다',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig4_pixiv_monthly.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig4_pixiv_monthly.png'))

## ⑤ 성별 (Step 7)
성별 지수는 성별마다 따로 100으로 맞춰지므로, 각 성별 안에서 기준어 대비 비율로만 비교합니다.

In [ ]:
# ---------- 5. 성별 ----------
G={}
for g,lab in [('female','여성'),('male','남성')]:
    x=dl(f'daily_gender_{g}.json').reindex(pd.date_range('2026-09-01','2026-10-22')).fillna(0).loc['2026-09-30':'2026-10-16']
    G[lab]={'AY_over_HP':round(x[AY].sum()/x[HP].sum(),2),'AY_over_MAL':round(x[AY].sum()/x[MAL].sum(),2)}
S['gender_0930_1016']=G
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
for a,key,ttl in [(ax[0],'AY_over_HP',"'악역영애 말포이' ÷ '해리포터'"),(ax[1],'AY_over_MAL',"'악역영애 말포이' ÷ '말포이'")]:
    v=[G['여성'][key],G['남성'][key]];a.bar(['여성','남성'],v,color=[C['light'],C['main']] if key=='AY_over_HP' else [C['light'],C['light']]);a.axhline(1,color=C['gray'],lw=.8,ls=':')
    for i,t in enumerate(v):a.text(i,t+0.03,f'{t}배',ha='center')
    a.set_title(ttl,fontsize=11)
fig.suptitle('⑤ 남성 검색에서는 팬픽 제목이 "해리포터"보다도 많이 검색됐다 (9/30~10/16 합계 비율)',x=0.01,ha='left',fontsize=12)
fig.text(0.01,-0.04,'자료: 네이버 데이터랩(성별 필터, 같은 요청 안의 비율). 성별 간 절대량 비교가 아니라 각 성별 안에서 두 검색어의 상대 크기',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig5_gender.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig5_gender.png'))

## ⑥ 5년 전과 비교 (Step 6)
주간 지수(말포이·악역영애 말포이 한 요청).

In [ ]:
# ---------- 6. 장기 주간 ----------
wk=dl('week_mal_ay.json').fillna(0);wk=wk.reindex(pd.date_range('2016-01-04',wk.index.max(),freq='W-MON')).fillna(0)
S['AY_weekly_max_2020_2025']=round(wk.loc['2020':'2025',AY].max(),3);S['AY_weekly_peak_2026']=round(wk[AY].max(),1)
fig,ax=plt.subplots(figsize=(12,3.8));r=wk.loc['2019':]
ax.plot(r.index,r[MAL],color=C['sub'],lw=1.4,label="'말포이'");ax.plot(r.index,r[AY],color=C['main'],lw=2,label="'악역영애 말포이'")
ax.axvspan(pd.Timestamp('2021-01-01'),pd.Timestamp('2021-12-31'),color=C['light'],alpha=.35);ax.text(pd.Timestamp('2021-02-01'),80,'2021년 (팬픽 완결)',fontsize=9,color=C['gray'])
ax.set_ylabel('주간 검색지수 (최고 주=100)');ax.legend(frameon=False,loc='upper left',bbox_to_anchor=(0,0.85))
ax.set_title(f"⑥ 완결 후 5년 동안 '악역영애 말포이' 검색은 거의 잡히지 않았다 (2020~2025 주간 최고 {S['AY_weekly_max_2020_2025']})\n네이버 데이터랩 주간 검색지수, 2019~2026",loc='left')
fig.text(0.01,-0.04,'자료: 네이버 데이터랩 검색어트렌드(주간). 검색량이 아주 적은 기간은 응답에서 빠져 0으로 채웠다. 2026-10-23 조회',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig6_longrun.png',bbox_inches='tight');plt.close()

from IPython.display import Image, display
display(Image(f'{FIG}/fig6_longrun.png'))

In [ ]:
import json
print(json.dumps(S, ensure_ascii=False, indent=1))

## 부록: 수집 코드 (기본은 실행하지 않음)

직접 다시 수집하려면 아래 셀의 `RUN = False`를 `True`로 바꾸세요. 네이버 데이터랩 키는 Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `NAVER_ID`, `NAVER_SECRET`으로 넣습니다. 키워드도구 검색수는 네이버 검색광고 화면이나 API로 따로 확인합니다.

In [ ]:
RUN = False
import json, time, datetime, urllib.request, urllib.parse
def datalab(body, cid, sec):
    req = urllib.request.Request('https://openapi.naver.com/v1/datalab/search', data=json.dumps(body).encode(), method='POST',
        headers={'X-Naver-Client-Id': cid, 'X-Naver-Client-Secret': sec, 'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req).read())
PIX_H = {'Referer': 'https://www.pixiv.net/', 'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0 Safari/537.36'}
def pixiv_total(tag, scd, ecd):
    q = urllib.parse.quote(tag)
    url = f'https://www.pixiv.net/ajax/search/artworks/{q}?word={q}&order=date_d&mode=all&p=1&s_mode=s_tag&type=all&scd={scd}&ecd={ecd}'
    return json.loads(urllib.request.urlopen(urllib.request.Request(url, headers=PIX_H)).read())['body']['illustManga']['total']
if RUN:
    from google.colab import userdata
    cid, sec = userdata.get('NAVER_ID'), userdata.get('NAVER_SECRET')
    G = lambda *gs: [{'groupName': g[0], 'keywords': list(g)} for g in gs]
    MAL, AY = ('말포이', '드레이코 말포이'), ('악역영애 말포이', '악역영애말포이')
    TS, HP = ('TS말포이', 'TS 말포이', '여자 말포이', '여자말포이'), ('해리포터', '해리 포터')
    daily = datalab({'startDate': '2026-08-01', 'endDate': '2026-10-22', 'timeUnit': 'date', 'keywordGroups': G(MAL, AY, TS, HP)}, cid, sec)
    print([r['title'] for r in daily['results']])
    d0 = datetime.date(2026, 9, 28); counts = {}
    for i in range(7):
        day = str(d0 + datetime.timedelta(days=i)); counts[day] = pixiv_total('Malfoid', day, day); time.sleep(0.8)
    print(counts)